# Categorize Responses in the Behavioural Set

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [ ]:


DATASET_SIZE = 40
ERROR_TAXONOMY = {
    "0": "Irrelevant Span Mislabeling",
    "1":"Missing Expected Entity",
    "2":"Tokenization Artifacts",
    "3":"Incorrect Polarity Assignment",
    "4":"BIO Sequencing Errors",
    "5":"Boundary Overreach",
    "6":"Boundary Undereach",
    "7":"Model Overgeneralization"
}

MODEL_NAME= "dmis-lab/biobert-base-cased-v1.1"
RUN_IDX ="run_2"# BEST RUN
VERSION = "v01"
# inference pipeline version may change as we improve and modify the pipeline
INFERENCE_PIPELINE_VERSION = "v01" 

# DATACLASSES TO DEFINE LOG ENTRY
@dataclass
class Spans:
    start: int
    end: int
    text: str
    label: str #Enum['O', 'SYMPTOM_POS', 'SYMPTOM_NEG', f'CONFLICT-{some variable}']
@dataclass
class LogEntry:
    date: datetime.datetime.now().strftime("%Y-%m-%d-%H:%M:%S")
    reviewer: Enum["HUMAN", "AI"]
    model: str=MODEL_NAME
    model_version: str =VERSION
    inference_pipeline_version: str = INFERENCE_PIPELINE_VERSION
    input_text: str
    tokens: List[str]
    token_level_labels: List[str]
    world_level_labels:List[str]
    predicted_spans: List[Spans]
    expected_entities: List[Spans]


In [ ]:
# Load the model
GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"

# Create a local directory path for the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Check if model already exists locally
if os.path.exists(LOCAL_MODEL_DIR) and os.path.isfile(os.path.join(LOCAL_MODEL_DIR, "config.json")):
    print(f"✅ Model found locally at {LOCAL_MODEL_DIR}")
    print("Skipping download from GCS.")
else:
    # Download the model directory from GCS if not found locally
    print(f"📥 Model not found locally. Downloading from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
    downloaded_path = download_from_gcs(
        gcs_path=GCS_MODEL_PATH,
        local_path=LOCAL_MODEL_DIR,
        bucket_name=BUCKET_NAME
    )
    if downloaded_path:
        print(f"✅ Download complete. Model saved to {LOCAL_MODEL_DIR}")

# Load the model
print(f"📂 Loading model from {LOCAL_MODEL_DIR}...")
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

# Error Categorization Loop